# Incident Commander — GRPO Training with Unsloth

This notebook fine-tunes **Llama-3.2-1B** using GRPO on the Incident Commander RL environment.

**What you'll need:**
- T4 GPU (free tier works for quick experiments)
- ~15 minutes for training

**References:**
- [Unsloth](https://github.com/unslothai/unsloth) — Memory-efficient fine-tuning
- [TRL GRPO](https://github.com/huggingface/trl) — RL training
- [Incident Commander Env](https://huggingface.co/spaces/YOUR-USERNAME/incident-commander)

## 1. Setup & Installation

In [ ]:
# Install all dependencies in one go
!pip install -q openenv-core[core] unsloth trl datasets peft accelerate bitsandbytes scipy

In [1]:
import os
import sys

# Clone or use the incident-commander repo
# Option A: From GitHub (uncomment if needed)
# !git clone https://github.com/YOUR-USERNAME/incident-commander.git
# Option B: If files are already in Colab, just add to path
sys.path.insert(0, '/content/incident-commander')

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: False


## 2. Import Environment

In [7]:
# Import our environment components directly from the repo
import sys
sys.path.insert(0, '/content/incident_commander')

from server.incident_commander_environment import IncidentCommanderEnvironment
from models import IncidentCommanderAction

# Quick sanity check
env = IncidentCommanderEnvironment(difficulty=1)
obs = env.reset()
print(f"Alert: {obs.alert_summary}")
print(f"Services: {obs.services_overview}")

ModuleNotFoundError: No module named 'server'

## 3. Load Base Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
import torch

# Patch Unsloth for GRPO support
PatchFastRL("GRPO", FastLanguageModel)
from trl import GRPOConfig, GRPOTrainer

# Configuration
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 2048
LORA_RANK = 16
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-5

print("Loading model via Unsloth (4-bit quantization)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6,
)
print("Model loaded!")

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA adapters applied!")

## 4. Define Reward Functions

In [ ]:
import re
from datasets import Dataset

SYSTEM_PROMPT = """You are an AI Incident Commander.
You must investigate microservice outages by using tools and reasoning causally.
Output your reasoning in <thought> tags, then lock your hypothesis and apply a fix.
Action format: <action>action_name:target_service</action>"""

def generate_initial_prompts(num_samples=100, difficulty=1):
    """Generate environment states for GRPO training."""
    prompts = []
    for _ in range(num_samples):
        env = IncidentCommanderEnvironment(difficulty=difficulty)
        obs = env.reset()
        prompt = f"System: {SYSTEM_PROMPT}\n\nObservation: {obs.alert_summary}\n"
        prompt += f"Services Overview: {obs.services_overview}\n\nWhat is your next action?"
        prompts.append({"prompt": prompt, "seed": env._ctx.scenario.seed if env._ctx else 0})
    return Dataset.from_list(prompts)

def parse_action(completion):
    """Extract action type and target from model output."""
    match = re.search(r"<action>(.*?):?(.*?)</action>", completion)
    if match:
        atype = match.group(1).strip()
        target = match.group(2).strip() if match.group(2) else "payment-service"
        return atype, target
    return None, None

def environment_reward_func(prompts, completions, **kwargs):
    """Evaluate completions using the Incident Commander environment."""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        env = IncidentCommanderEnvironment(difficulty=1)
        env.reset()
        action_type, target = parse_action(completion)
        if not action_type:
            rewards.append(0.0)
            continue
        try:
            obs = env.step(IncidentCommanderAction(action_type=action_type, target_service=target))
            rewards.append(obs.reward)
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

def format_reward_func(prompts, completions, **kwargs):
    """Bonus reward for using proper <thought> and <action> tags."""
    rewards = []
    for completion in completions:
        score = 0.0
        if "<thought>" in completion and "</thought>" in completion:
            score += 0.1
        if "<action>" in completion and "</action>" in completion:
            score += 0.1
        rewards.append(score)
    return rewards

## 5. Generate Training Dataset

In [ ]:
print("Generating curriculum tasks (Difficulty 1 - Easy)...")
dataset = generate_initial_prompts(num_samples=250, difficulty=1)
print(f"Dataset created with {len(dataset)} samples")
print(f"Example prompt: {dataset[0]['prompt'][:200]}...")

## 6. Initialize GRPO Trainer

In [ ]:
training_args = GRPOConfig(
    output_dir="outputs/commander_grpo",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_prompt_length=1024,
    max_completion_length=512,
    num_train_epochs=1,
    logging_steps=10,
    beta=0.1,  # KL penalty
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[environment_reward_func, format_reward_func],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("GRPOTrainer initialized!")

## 7. Start Training

In [ ]:
print("Starting GRPO Training...")
print("=" * 50)
trainer.train()
print("=" * 50)
print("Training complete!")

## 8. Save the Trained Model

In [ ]:
import os
os.makedirs("outputs/commander_final", exist_ok=True)

model.save_pretrained_merged("outputs/commander_final", tokenizer, save_method="merged_16bit")
print("Model saved to outputs/commander_final")

# Download for local use
from google.colab import files
!tar -czvf commander_final.tar.gz outputs/commander_final
files.download("commander_final.tar.gz")

## 9. Evaluate the Model

In [ ]:
# Load model for inference
FastLanguageModel.for_inference(model)

def run_evaluation_episode(difficulty=1, max_steps=50):
    """Run one episode with the trained model."""
    env = IncidentCommanderEnvironment(difficulty=difficulty)
    obs = env.reset()
    total_reward = 0

    for step in range(max_steps):
        prompt = f"System: {SYSTEM_PROMPT}\n\nObservation: {obs.alert_summary}\n"
        if obs.last_action_result:
            prompt += f"Previous: {obs.last_action_result}\n"
        prompt += f"Services: {obs.services_overview}\n\nWhat is your next action?"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
        completion = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

        action_type, target = parse_action(completion)
        if not action_type:
            print(f"Step {step+1}: Invalid format, skipping")
            continue

        obs = env.step(IncidentCommanderAction(action_type=action_type, target_service=target))
        total_reward = obs.reward
        print(f"Step {step+1}: {action_type} on {target} → reward={obs.reward:.3f}")

        if obs.done:
            print(f"\nEpisode resolved! {obs.last_action_result}")
            break

    if not obs.done:
        print(f"\nEpisode timed out (>{max_steps} steps)")

    return total_reward

# Run a test episode
print("\n" + "=" * 50)
final_reward = run_evaluation_episode(difficulty=1)
print(f"\nFinal cumulative reward: {final_reward:.3f}")

## 10. Push Model to Hugging Face (Optional)

In [ ]:
# Option A: Push to Hugging Face Hub
# !pip install huggingface_hub
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="outputs/commander_final",
#     repo_id="your-username/incident-commander-model",
#     repo_type="model",
# )

# Option B: Download the saved model
print("Model files saved locally in outputs/commander_final")
print("You can download them from the Colab file browser (left sidebar)")

---

## Quick Reference: Action Schema

| Action | Target Required | Extra Fields | Description |
|--------|----------------|-------------|-------------|
| `read_logs` | Yes | `time_range_minutes` | Read service logs |
| `read_metrics` | Yes | — | Read service metrics |
| `read_deployment_history` | No | — | View deployment history |
| `read_dependency_graph` | No | — | View service graph |
| `identify_cause` | Yes | `hypothesis` | Lock root cause |
| `restart_pod` | Yes | — | Restart a pod |
| `rollback` | Yes | — | Rollback deployment |
| `scale_up` | Yes | — | Scale up pods |
| `hotfix` | Yes | — | Apply hotfix |
| `escalate` | No | `justification` | Escalate to on-call |
| `monitor_recovery` | No | — | Observe post-fix status |
| `resolve` | No | `justification` | Close the incident |

### Root Cause → Fix Mapping

| Root Cause | Correct Fix |
|------------|-------------|
| `memory_limit_too_low` | `scale_up` |
| `bad_deployment` | `rollback` |
| `connection_pool_exhausted` | `hotfix` |
| `traffic_spike` | `scale_up` |
| `dependency_failure` | `restart_pod` |
| `config_error` | `hotfix` |
| `redis_down` | `restart_pod` |
| `certificate_expired` | `hotfix` |